# Evaluación Parcial 2: Machine Learning supervisado

**Asignatura:** MLY0100 | **Integrantes:** Christian Sandoval y Nicolás Vega | **Fecha:** 5 de noviembre de 2026

**Dataset:** Properati limpio de EV1. **CRISP-DM:** fases 4 (Modelado) y 5 (Evaluación).

Se desarrollan dos tareas: regresión para estimar price en USD y clasificación de property_type en Departamento, Casa, PH y Otros. Las conclusiones se generan a partir de las métricas ejecutadas.

## 1. Contexto y objetivos

El caso estudia publicaciones de propiedades en venta de Argentina. En regresión se estima un valor monetario continuo (price); en clasificación se asigna una categoría discreta (property_type). Se comparan modelos supervisados y su generalización. property_type solo se usa como feature en regresión. title y description se excluyen de clasificación para evitar data leakage.

## 2. Librerías

Se utilizan Pandas, NumPy, Matplotlib, Seaborn, Scikit-Learn e imbalanced-learn para preparar, modelar y evaluar los datos.

In [ ]:
import os
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.model_selection import (
    train_test_split, KFold, StratifiedKFold, cross_validate
)
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline
from IPython.display import display, Markdown

warnings.filterwarnings('ignore', category=FutureWarning)
RANDOM_STATE = 42
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

print('Librerías cargadas correctamente.')

## 3. Carga del dataset limpio de EV1

Se busca el CSV preparado. Si no existe, se reconstruye desde el ZIP aplicando la limpieza final de EV1: medianas agrupadas e identificación conjunta de outliers mediante IQR y Z-score. Se conservan variables originales; los escaladores se ajustan nuevamente después de la división train/test para evitar fuga de información.

In [ ]:
COLUMNAS_MODELO = [
    'lat', 'lon', 'l2', 'l3', 'rooms', 'bedrooms', 'bathrooms',
    'surface_total', 'surface_covered', 'price', 'property_type'
]
COLUMNAS_NUMERICAS_EV1 = [
    'rooms', 'bedrooms', 'bathrooms', 'surface_total',
    'surface_covered', 'price', 'lat', 'lon'
]


def localizar_archivo(nombre):
    candidatos = [
        Path.cwd() / nombre,
        Path.cwd().parent / nombre,
        Path('/content') / nombre,
        Path('/content/drive/MyDrive') / nombre
    ]
    for ruta in candidatos:
        if ruta.exists():
            return ruta
    return None


ruta_preparado = localizar_archivo('DS1-18-Datos-Properati-preparado.csv')
ruta_zip = localizar_archivo('Evaluación Parcial 1 - Dataset.zip')

if ruta_preparado is not None:
    print(f'Leyendo dataset preparado de EV1: {ruta_preparado}')
    df = pd.read_csv(ruta_preparado, usecols=lambda c: c in COLUMNAS_MODELO, low_memory=False)
else:
    if ruta_zip is None:
        raise FileNotFoundError(
            'No se encontró el CSV preparado ni el ZIP original. '
            'Coloca "Evaluación Parcial 1 - Dataset.zip" en la raíz del repositorio '
            'o ejecuta primero notebook/evaluacion_parcial_1.ipynb.'
        )
    print(f'No existe el CSV preparado. Se reconstruye la limpieza final de EV1 desde: {ruta_zip}')
    with zipfile.ZipFile(ruta_zip) as archivo_zip:
        nombre_csv = next(n for n in archivo_zip.namelist() if n.endswith('DS1-18-Datos-Properati.csv'))
        with archivo_zip.open(nombre_csv) as archivo_csv:
            df = pd.read_csv(archivo_csv, usecols=COLUMNAS_MODELO, low_memory=False)

    # Imputación final utilizada en EV1: medianas por tipo de propiedad y localidad.
    for columna in ['surface_total', 'surface_covered', 'bathrooms']:
        mediana_grupo = df.groupby('property_type', observed=False)[columna].transform('median')
        df[columna] = df[columna].fillna(mediana_grupo)

    for columna in ['lat', 'lon']:
        mediana_grupo = df.groupby('l3', observed=False)[columna].transform('median')
        df[columna] = df[columna].fillna(mediana_grupo)

    # Criterio de EV1: eliminar solo los registros marcados por IQR y Z-score a la vez.
    mascara_extremos = np.zeros(len(df), dtype=bool)
    for columna in COLUMNAS_NUMERICAS_EV1:
        q1 = df[columna].quantile(0.25)
        q3 = df[columna].quantile(0.75)
        iqr = q3 - q1
        mascara_iqr = ((df[columna] < q1 - 1.5 * iqr) | (df[columna] > q3 + 1.5 * iqr)).to_numpy()
        z_score = np.abs(stats.zscore(df[columna].to_numpy(), nan_policy='omit'))
        mascara_z = z_score > 3
        mascara_extremos |= mascara_iqr & mascara_z
    df = df.loc[~mascara_extremos].copy()

# Tipos consistentes para el análisis y el preprocesamiento.
for columna in ['l2', 'l3', 'property_type']:
    df[columna] = df[columna].astype('string').fillna('Sin dato').astype(str)

for columna in ['lat', 'lon', 'rooms', 'bedrooms', 'bathrooms', 'surface_total', 'surface_covered', 'price']:
    df[columna] = pd.to_numeric(df[columna], errors='coerce')

df = df.dropna(subset=COLUMNAS_MODELO).copy()
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=COLUMNAS_MODELO).copy()

print('Dimensiones del dataset utilizado:', df.shape)
print('Valores faltantes en columnas de modelado:', int(df[COLUMNAS_MODELO].isna().sum().sum()))
print('Tipos de propiedad originales:')
display(df['property_type'].value_counts().to_frame('cantidad'))

## 4. Targets y distribución

price se modela con log1p por su asimetría y las métricas se devuelven a USD. Para clasificación se agrupan Lote, Oficina, Local comercial, Casa de campo, Depósito, Cochera y Otro en Otros.

In [ ]:
df['clase'] = np.where(
    df['property_type'].isin(['Departamento', 'Casa', 'PH']),
    df['property_type'],
    'Otros'
)

print('Target de regresión:')
print('dtype:', df['price'].dtype)
print('Cantidad de valores únicos:', df['price'].nunique())
print(df['price'].describe())

print('\nTarget de clasificación:')
print('dtype:', df['clase'].dtype)
print('Cantidad de clases:', df['clase'].nunique())
display(df['clase'].value_counts().to_frame('cantidad'))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(df['price'], bins=50, edgecolor='black')
ax[0].set_title('Distribución del precio')
ax[0].set_xlabel('Precio (USD)')
ax[0].set_ylabel('Frecuencia')
ax[1].hist(np.log1p(df['price']), bins=50, edgecolor='black')
ax[1].set_title('Distribución de log1p(price)')
ax[1].set_xlabel('log1p(Precio)')
ax[1].set_ylabel('Frecuencia')
plt.tight_layout()
plt.show()

frecuencias = df['clase'].value_counts()
porcentajes = (df['clase'].value_counts(normalize=True) * 100).round(2)
display(pd.DataFrame({'cantidad': frecuencias, 'porcentaje': porcentajes}))

plt.figure(figsize=(8, 4))
plt.bar(frecuencias.index.astype(str), frecuencias.values)
plt.title('Distribución de las cuatro clases')
plt.xlabel('Tipo de propiedad agrupado')
plt.ylabel('Cantidad de publicaciones')
plt.xticks(rotation=15)
plt.show()

## 5. División train/test

Se utilizan muestras reproducibles de hasta 12.000 registros para regresión y 8.000 para clasificación. La muestra de clasificación y su división 80/20 son estratificadas. Los cuatro modelos usan la misma división.

In [ ]:
# REGRESIÓN: muestra aleatoria reproducible y división 80/20.
N_REGRESION = min(12_000, len(df))
df_reg = df.sample(n=N_REGRESION, random_state=RANDOM_STATE).copy()

features_reg_numericas = ['rooms', 'bathrooms', 'surface_total', 'surface_covered', 'lat', 'lon']
features_reg_categoricas = ['l2', 'l3', 'property_type']
features_reg = features_reg_numericas + features_reg_categoricas
X_reg = df_reg[features_reg].copy()
y_reg = df_reg['price'].astype(float).copy()

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=RANDOM_STATE
)

# CLASIFICACIÓN: muestra estratificada de hasta 8.000 registros.
N_CLASIFICACION = min(8_000, len(df))
df_clf, _ = train_test_split(
    df, train_size=N_CLASIFICACION, stratify=df['clase'], random_state=RANDOM_STATE
)

features_clf_numericas = ['rooms', 'bathrooms', 'surface_total', 'surface_covered', 'lat', 'lon']
features_clf_categoricas = ['l2', 'l3']
features_clf = features_clf_numericas + features_clf_categoricas
X_clf = df_clf[features_clf].copy()
y_clf = df_clf['clase'].copy()

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.20, random_state=RANDOM_STATE, stratify=y_clf
)

print('Regresión | train:', X_train_reg.shape, '| test:', X_test_reg.shape)
print('Clasificación | train:', X_train_clf.shape, '| test:', X_test_clf.shape)
print('\nDistribución de clasificación en entrenamiento:')
display(y_train_clf.value_counts().to_frame('cantidad_train'))
print('Distribución de clasificación en prueba:')
display(y_test_clf.value_counts().to_frame('cantidad_test'))

## 6. Correlación

La correlación se analiza en entrenamiento. Pearson resume relaciones lineales y Spearman relaciones monótonas. Se revisan pares con |r| > 0,8 y la relación de los atributos numéricos con el precio.

In [ ]:
# bedrooms se incorpora al análisis exploratorio para justificar su descarte,
# pero no se entrega a los modelos porque presenta alta correlación con rooms.
features_corr_numericas = [
    'rooms', 'bedrooms', 'bathrooms', 'surface_total',
    'surface_covered', 'lat', 'lon'
]
df_corr_train = df_reg.loc[X_train_reg.index, features_corr_numericas].copy()
df_corr_train['price'] = y_train_reg

corr_pearson = df_corr_train.corr(method='pearson')
corr_spearman = df_corr_train.corr(method='spearman')

fig, ax = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(corr_pearson, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax[0])
ax[0].set_title('Correlación de Pearson (entrenamiento)')
sns.heatmap(corr_spearman, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax[1])
ax[1].set_title('Correlación de Spearman (entrenamiento)')
plt.tight_layout()
plt.show()

print('Correlación entre rooms y bedrooms en el conjunto de entrenamiento:')
print(f"Pearson: {corr_pearson.loc['rooms', 'bedrooms']:.3f}")
print(f"Spearman: {corr_spearman.loc['rooms', 'bedrooms']:.3f}")

correlacion_precio = corr_spearman['price'].drop('price').sort_values(key=np.abs, ascending=False)
print('Correlación de Spearman de cada variable numérica con price:')
display(correlacion_precio.to_frame('spearman_con_price'))

pares_fuertes = []
for i, variable_a in enumerate(features_corr_numericas):
    for variable_b in features_corr_numericas[i + 1:]:
        valor = corr_pearson.loc[variable_a, variable_b]
        if abs(valor) > 0.8:
            pares_fuertes.append((variable_a, variable_b, valor))

if pares_fuertes:
    display(pd.DataFrame(pares_fuertes, columns=['feature_1', 'feature_2', 'pearson']).sort_values('pearson'))
else:
    print('No se encontraron pares con |Pearson| > 0,8 entre las variables analizadas.')

# Para clasificación también se revisan las medianas por clase, incluyendo
# bedrooms como variable de diagnóstico, aunque no se usa para entrenar.
df_clf_train_eda = X_train_clf.copy()
df_clf_train_eda['bedrooms'] = df_clf.loc[X_train_clf.index, 'bedrooms']
df_clf_train_eda['clase'] = y_train_clf
features_clf_eda = features_clf_numericas + ['bedrooms']
medianas_por_clase = df_clf_train_eda.groupby('clase')[features_clf_eda].median().round(2)
display(medianas_por_clase)

plt.figure(figsize=(9, 5))
sns.boxplot(data=df_clf_train_eda, x='clase', y='surface_total', showfliers=False)
plt.title('Superficie total por clase de propiedad (entrenamiento)')
plt.xlabel('Clase')
plt.ylabel('Superficie total (m²)')
plt.show()

### Decisiones de selección de features

- Se conserva **rooms** y se descarta **bedrooms** en regresión y clasificación para reducir multicolinealidad. En el conjunto de entrenamiento, Pearson entre ambas es aproximadamente **0,887** y Spearman **0,921**. Son asociaciones altas y describen dimensiones cercanas del tamaño de la vivienda.
- Se conservan **surface_total** y **surface_covered**: describen superficie total y cubierta. En la matriz de entrenamiento no superan entre sí el umbral de referencia |r| > 0,8, por lo que aportan información complementaria.
- Se conservan **bathrooms**, **lat** y **lon** por su relación con características físicas y ubicación. Pearson y Spearman permiten observar relaciones lineales y monótonas con price.
- **l2** y **l3** se codifican con One-Hot Encoding. Son variables nominales, por lo que no corresponde asignarles un orden numérico artificial.
- Para regresión se incluye **property_type**, porque el tipo de inmueble es conocido al estimar su precio. Para clasificación se excluye porque es la variable objetivo.
- Se descartan **title** y **description** para clasificación por riesgo de data leakage; también se descartan fechas sin transformar, columnas constantes (l1, currency, operation_type) y columnas escaladas en EV1, que se ajustaron antes de separar train/test.
- **price_minmax** se excluye siempre de los features porque es una transformación directa del target price.

Estas decisiones reducen redundancia y evitan entregar al modelo información que no estaría disponible de forma legítima al realizar una predicción.

## 7. Preparación para regresión

ColumnTransformer imputa y escala numéricas, e imputa y codifica categóricas. El preprocesamiento vive dentro del Pipeline y aprende solo de entrenamiento, también en cada fold.

In [ ]:
preprocesador_reg = ColumnTransformer(
    transformers=[
        ('numericas', Pipeline(steps=[
            ('imputacion', SimpleImputer(strategy='median')),
            ('escalado', StandardScaler())
        ]), features_reg_numericas),
        ('categoricas', Pipeline(steps=[
            ('imputacion', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), features_reg_categoricas)
    ],
    remainder='drop'
)

print('Features numéricos de regresión:', features_reg_numericas)
print('Features categóricos de regresión:', features_reg_categoricas)
print('El escalado y el One-Hot Encoding se ajustarán dentro del Pipeline usando solo X_train_reg.')

## 8. Modelos de regresión (Fase 4 - Modelado)

Se comparan dos algoritmos permitidos en la guía:

1. **Regresión lineal múltiple:** modelo base que estima una relación lineal entre las características y el logaritmo del precio.
2. **KNN Regressor:** modelo basado en vecinos cercanos que puede representar relaciones no lineales. Requiere escalado porque utiliza distancias. Se utiliza k=7 y ponderación uniforme (weights='uniform'), de modo que los siete vecinos contribuyen por igual. Esta configuración evita que un vecino de distancia cero domine el resultado cuando se evalúa una observación del propio entrenamiento.

En ambos se transforma price mediante log1p para reducir el efecto de la asimetría. TransformedTargetRegressor devuelve las predicciones a USD mediante expm1, de modo que las métricas se interpretan en la escala original.

In [ ]:
def crear_modelo_regresion(modelo):
    pipeline = Pipeline(steps=[
        ('preprocesamiento', preprocesador_reg),
        ('modelo', modelo)
    ])
    return TransformedTargetRegressor(
        regressor=pipeline,
        func=np.log1p,
        inverse_func=np.expm1
    )

modelos_regresion = {
    'Regresión lineal múltiple': crear_modelo_regresion(LinearRegression()),
    'KNN Regressor (k=7, uniform)': crear_modelo_regresion(KNeighborsRegressor(n_neighbors=7, weights='uniform'))
}

resultados_regresion = []
for nombre, modelo in modelos_regresion.items():
    print(f'Entrenando: {nombre}')
    modelo.fit(X_train_reg, y_train_reg)
    pred_train = modelo.predict(X_train_reg)
    pred_test = modelo.predict(X_test_reg)
    resultados_regresion.append({
        'modelo': nombre,
        'MAE_train_USD': mean_absolute_error(y_train_reg, pred_train),
        'RMSE_train_USD': np.sqrt(mean_squared_error(y_train_reg, pred_train)),
        'R2_train': r2_score(y_train_reg, pred_train),
        'MAE_test_USD': mean_absolute_error(y_test_reg, pred_test),
        'RMSE_test_USD': np.sqrt(mean_squared_error(y_test_reg, pred_test)),
        'R2_test': r2_score(y_test_reg, pred_test)
    })

tabla_regresion = pd.DataFrame(resultados_regresion).set_index('modelo')
display(tabla_regresion.sort_values('MAE_test_USD'))

## 9. Métricas y selección del modelo de regresión (Fase 5 - Evaluación)

- **MAE:** error absoluto medio; indica cuántos USD se equivoca el modelo en promedio y es fácil de interpretar.
- **RMSE:** raíz del error cuadrático medio; también se expresa en USD, pero penaliza más los errores grandes.
- **R²:** proporción de variabilidad del precio explicada por el modelo. Puede ser negativo si predice peor que una referencia basada en la media.

La selección considera el desempeño en prueba y la brecha entre entrenamiento y prueba. No se elige únicamente el modelo que mejor se ajusta a los datos usados para entrenar. En KNN se usa ponderación uniforme para reducir el efecto del auto-vecino en la evaluación de entrenamiento.

In [ ]:
tabla_regresion['brecha_MAE_USD'] = abs(tabla_regresion['MAE_test_USD'] - tabla_regresion['MAE_train_USD'])
mejor_regresion = tabla_regresion.sort_values(['MAE_test_USD', 'brecha_MAE_USD']).iloc[0]
nombre_mejor_regresion = mejor_regresion.name

display(tabla_regresion.sort_values('MAE_test_USD'))
print(
    f'Modelo seleccionado por menor MAE en prueba: {nombre_mejor_regresion}. '
    f'MAE de prueba = USD {mejor_regresion["MAE_test_USD"]:,.2f}; '
    f'RMSE de prueba = USD {mejor_regresion["RMSE_test_USD"]:,.2f}; '
    f'R² de prueba = {mejor_regresion["R2_test"]:.3f}; '
    f'brecha de MAE train-test = USD {mejor_regresion["brecha_MAE_USD"]:,.2f}.'
)

fig, ax = plt.subplots(figsize=(8, 4))
tabla_regresion[['MAE_train_USD', 'MAE_test_USD']].plot(kind='bar', ax=ax)
ax.set_title('Comparación de MAE en entrenamiento y prueba')
ax.set_ylabel('MAE (USD)')
ax.set_xlabel('Modelo')
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

### Validación cruzada para regresión

Se utiliza K-Fold de 3 particiones sobre una submuestra del conjunto de entrenamiento. cross_validate(..., return_train_score=True) devuelve medias y desviaciones estándar de entrenamiento y validación para MAE y R². El preprocesamiento y la transformación del target están dentro del estimador, por lo que se vuelven a ajustar dentro de cada fold. La selección se contrasta especialmente con el MAE de validación, porque representa observaciones que el modelo no utilizó para ajustarse.

In [ ]:
N_CV_REG = min(2_500, len(X_train_reg))
X_cv_reg = X_train_reg.sample(n=N_CV_REG, random_state=RANDOM_STATE)
y_cv_reg = y_train_reg.loc[X_cv_reg.index]
cv_reg = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

resultados_cv_reg = []
for nombre, modelo in modelos_regresion.items():
    cv_resultado = cross_validate(
        modelo, X_cv_reg, y_cv_reg,
        cv=cv_reg,
        scoring={'MAE': 'neg_mean_absolute_error', 'R2': 'r2'},
        return_train_score=True,
        n_jobs=1
    )
    resultados_cv_reg.append({
        'modelo': nombre,
        'MAE_CV_train_media_USD': -cv_resultado['train_MAE'].mean(),
        'MAE_CV_train_std_USD': cv_resultado['train_MAE'].std(),
        'MAE_CV_validacion_media_USD': -cv_resultado['test_MAE'].mean(),
        'MAE_CV_validacion_std_USD': cv_resultado['test_MAE'].std(),
        'R2_CV_validacion_media': cv_resultado['test_R2'].mean(),
        'R2_CV_validacion_std': cv_resultado['test_R2'].std()
    })

tabla_cv_regresion = pd.DataFrame(resultados_cv_reg).set_index('modelo')
display(tabla_cv_regresion)

## 10. Modelos de clasificación

Se implementan los cuatro algoritmos permitidos: Regresión Logística, KNN, SVM lineal y Decision Tree. Se excluye property_type de los features porque es el target. Se usa la misma división estratificada.

In [ ]:
preprocesador_clf = ColumnTransformer(
    transformers=[
        ('numericas', Pipeline(steps=[
            ('imputacion', SimpleImputer(strategy='median')),
            ('escalado', StandardScaler())
        ]), features_clf_numericas),
        ('categoricas', Pipeline(steps=[
            ('imputacion', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), features_clf_categoricas)
    ],
    remainder='drop'
)

modelos_clasificacion = {
    'Regresión Logística': Pipeline(steps=[
        ('preprocesamiento', preprocesador_clf),
        ('modelo', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]),
    'KNN Classifier (k=7)': Pipeline(steps=[
        ('preprocesamiento', preprocesador_clf),
        ('modelo', KNeighborsClassifier(n_neighbors=7, weights='distance'))
    ]),
    'SVM lineal': Pipeline(steps=[
        ('preprocesamiento', preprocesador_clf),
        ('modelo', SVC(kernel='linear', C=1.0, probability=True, random_state=RANDOM_STATE))
    ]),
    'Decision Tree': Pipeline(steps=[
        ('preprocesamiento', preprocesador_clf),
        ('modelo', DecisionTreeClassifier(max_depth=12, min_samples_leaf=10, random_state=RANDOM_STATE))
    ])
}

print('Features numéricos:', features_clf_numericas)
print('Features categóricos:', features_clf_categoricas)
print('Se entrenarán los cuatro modelos permitidos sobre la misma división estratificada.')

### Métricas y entrenamiento

Se calculan accuracy, precision, recall, F1 macro, F1 ponderado y ROC-AUC multiclase One-vs-Rest con probabilidades.

In [ ]:
ETIQUETAS = ['Departamento', 'Casa', 'PH', 'Otros']


def metricas_clasificacion(modelo, X, y):
    pred = modelo.predict(X)
    proba = modelo.predict_proba(X)
    return {
        'accuracy': accuracy_score(y, pred),
        'precision_macro': precision_score(y, pred, average='macro', zero_division=0),
        'recall_macro': recall_score(y, pred, average='macro', zero_division=0),
        'f1_macro': f1_score(y, pred, average='macro', zero_division=0),
        'f1_weighted': f1_score(y, pred, average='weighted', zero_division=0),
        'roc_auc_ovr_macro': roc_auc_score(
            y, proba, labels=modelo.classes_, multi_class='ovr', average='macro'
        )
    }


resultados_clasificacion = []
for nombre, modelo in modelos_clasificacion.items():
    print(f'Entrenando: {nombre}')
    modelo.fit(X_train_clf, y_train_clf)
    m_train = metricas_clasificacion(modelo, X_train_clf, y_train_clf)
    m_test = metricas_clasificacion(modelo, X_test_clf, y_test_clf)
    fila = {'modelo': nombre}
    fila.update({f'{k}_train': v for k, v in m_train.items()})
    fila.update({f'{k}_test': v for k, v in m_test.items()})
    resultados_clasificacion.append(fila)

tabla_clasificacion = pd.DataFrame(resultados_clasificacion).set_index('modelo')
display(tabla_clasificacion.sort_values('f1_macro_test', ascending=False))

### Reportes y matrices de confusión

El reporte presenta métricas por clase y promedios macro/weighted. Las filas de la matriz son clases reales y las columnas clases predichas.

In [ ]:
matrices_confusion = {}
for nombre, modelo in modelos_clasificacion.items():
    pred_test = modelo.predict(X_test_clf)
    print(f'\n===== {nombre} =====')
    print(classification_report(y_test_clf, pred_test, labels=ETIQUETAS, zero_division=0))
    cm = confusion_matrix(y_test_clf, pred_test, labels=ETIQUETAS)
    matrices_confusion[nombre] = cm
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=ETIQUETAS).plot(
        ax=ax, cmap='Blues', colorbar=False, values_format='d', xticks_rotation=25
    )
    ax.set_title(f'Matriz de confusión: {nombre}')
    plt.tight_layout()
    plt.show()

### Interpretación de precision, recall y errores por clase

classification_report y la matriz de confusión permiten explicar qué categorías reconoce el modelo y cuáles confunde. Las filas de la matriz son las clases reales y las columnas las predichas. Para la clase **Otros**, los verdaderos positivos (TP) son los registros Otros correctamente identificados; los falsos positivos (FP) son propiedades de otras clases predichas como Otros; los falsos negativos (FN) son propiedades Otros clasificadas en otra categoría.

- precision = TP / (TP + FP): de todas las publicaciones predichas como Otros, qué proporción realmente pertenece a Otros.
- recall = TP / (TP + FN): de todas las publicaciones que realmente son Otros, qué proporción fue detectada.

La siguiente celda calcula estas cantidades y resume, para cada clase, el error de clasificación más frecuente. Se usa Decision Tree como referencia porque la selección final pondera F1 macro y generalización, no solo el máximo F1 aislado.

In [ ]:
mejor_clf_base = 'Decision Tree'
cm_mejor = matrices_confusion[mejor_clf_base]

# Resumen de aciertos y principal confusión de cada clase.
resumen_confusiones = []
for i, clase_real in enumerate(ETIQUETAS):
    soporte = int(cm_mejor[i, :].sum())
    aciertos = int(cm_mejor[i, i])
    errores = cm_mejor[i, :].copy()
    errores[i] = 0
    if errores.sum() > 0:
        j = int(np.argmax(errores))
        confusion_principal = ETIQUETAS[j]
        cantidad_confusion = int(errores[j])
    else:
        confusion_principal = 'Ninguna'
        cantidad_confusion = 0
    resumen_confusiones.append({
        'clase_real': clase_real,
        'casos_reales': soporte,
        'aciertos': aciertos,
        'recall': aciertos / soporte if soporte else 0,
        'principal_confusion': confusion_principal,
        'cantidad_confundida': cantidad_confusion
    })

tabla_confusiones = pd.DataFrame(resumen_confusiones).set_index('clase_real')
display(tabla_confusiones.round(3))

idx_otros = ETIQUETAS.index('Otros')
TP = int(cm_mejor[idx_otros, idx_otros])
FP = int(cm_mejor[:, idx_otros].sum() - TP)
FN = int(cm_mejor[idx_otros, :].sum() - TP)
precision_otros = TP / (TP + FP) if (TP + FP) else 0
recall_otros = TP / (TP + FN) if (TP + FN) else 0

print('Clasificador de referencia:', mejor_clf_base)
print(f'TP (Otros): {TP}; FP (Otros): {FP}; FN (Otros): {FN}')
print(f'Precision de Otros calculada desde la matriz: {precision_otros:.3f}')
print(f'Recall de Otros calculado desde la matriz: {recall_otros:.3f}')

## 11. Balance de clases

RandomOverSampler se aplica solo al entrenamiento. El conjunto de prueba conserva su distribución original. El muestreador está dentro de un imblearn Pipeline para que también opere dentro de cada fold de validación.

In [ ]:
ros = RandomOverSampler(random_state=RANDOM_STATE)
_, y_train_balanceado = ros.fit_resample(X_train_clf, y_train_clf)

distribucion_balance = pd.DataFrame({
    'antes_balance': y_train_clf.value_counts(),
    'despues_balance': pd.Series(y_train_balanceado).value_counts()
}).fillna(0).astype(int).reindex(ETIQUETAS)
display(distribucion_balance)

ax = distribucion_balance.plot(kind='bar', figsize=(9, 4))
ax.set_title('Distribución de clases antes y después de RandomOverSampler')
ax.set_xlabel('Clase')
ax.set_ylabel('Cantidad en entrenamiento')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

modelo_arbol_balanceado = ImbPipeline(steps=[
    ('preprocesamiento', preprocesador_clf),
    ('balanceo', RandomOverSampler(random_state=RANDOM_STATE)),
    ('modelo', DecisionTreeClassifier(max_depth=12, min_samples_leaf=10, random_state=RANDOM_STATE))
])
modelo_arbol_balanceado.fit(X_train_clf, y_train_clf)

metricas_arbol_balanceado_train = metricas_clasificacion(modelo_arbol_balanceado, X_train_clf, y_train_clf)
metricas_arbol_balanceado_test = metricas_clasificacion(modelo_arbol_balanceado, X_test_clf, y_test_clf)

pred_balanceado = modelo_arbol_balanceado.predict(X_test_clf)
print(classification_report(y_test_clf, pred_balanceado, labels=ETIQUETAS, zero_division=0))

cm_balanceado = confusion_matrix(y_test_clf, pred_balanceado, labels=ETIQUETAS)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm_balanceado, display_labels=ETIQUETAS).plot(
    ax=ax, cmap='Blues', colorbar=False, values_format='d', xticks_rotation=25
)
ax.set_title('Matriz de confusión: Decision Tree + RandomOverSampler')
plt.tight_layout()
plt.show()

comparacion_balance = pd.DataFrame([
    {
        'modelo': 'Decision Tree sin balance',
        'f1_macro_test': tabla_clasificacion.loc['Decision Tree', 'f1_macro_test'],
        'recall_Otros_test': recall_score(y_test_clf, modelos_clasificacion['Decision Tree'].predict(X_test_clf), labels=['Otros'], average='macro', zero_division=0)
    },
    {
        'modelo': 'Decision Tree + RandomOverSampler',
        'f1_macro_test': metricas_arbol_balanceado_test['f1_macro'],
        'recall_Otros_test': recall_score(y_test_clf, pred_balanceado, labels=['Otros'], average='macro', zero_division=0)
    }
]).set_index('modelo')
display(comparacion_balance)

## 12. Validación cruzada y generalización

Stratified K-Fold compara F1 macro y accuracy de entrenamiento/validación. Una brecha alta sugiere overfitting; resultados bajos en ambos conjuntos sugieren underfitting.

In [ ]:
N_CV_CLF = min(2_500, len(X_train_clf))
X_cv_clf, _ = train_test_split(
    X_train_clf, train_size=N_CV_CLF,
    stratify=y_train_clf, random_state=RANDOM_STATE
)
y_cv_clf = y_train_clf.loc[X_cv_clf.index]
cv_clf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

modelos_cv_clf = dict(modelos_clasificacion)
modelos_cv_clf['Decision Tree + RandomOverSampler'] = modelo_arbol_balanceado

resultados_cv_clf = []
for nombre, modelo in modelos_cv_clf.items():
    print(f'Validación cruzada: {nombre}')
    cv_resultado = cross_validate(
        modelo, X_cv_clf, y_cv_clf,
        cv=cv_clf,
        scoring={'f1_macro': 'f1_macro', 'accuracy': 'accuracy'},
        return_train_score=True,
        n_jobs=1
    )
    resultados_cv_clf.append({
        'modelo': nombre,
        'F1_macro_train_CV_media': cv_resultado['train_f1_macro'].mean(),
        'F1_macro_train_CV_std': cv_resultado['train_f1_macro'].std(),
        'F1_macro_validacion_CV_media': cv_resultado['test_f1_macro'].mean(),
        'F1_macro_validacion_CV_std': cv_resultado['test_f1_macro'].std(),
        'accuracy_validacion_CV_media': cv_resultado['test_accuracy'].mean(),
        'accuracy_validacion_CV_std': cv_resultado['test_accuracy'].std()
    })

tabla_cv_clasificacion = pd.DataFrame(resultados_cv_clf).set_index('modelo')
display(tabla_cv_clasificacion.sort_values('F1_macro_validacion_CV_media', ascending=False))

### Diagnóstico de overfitting y underfitting

Se comparan tres complejidades del mismo algoritmo Decision Tree: un árbol muy simple (max_depth=1), un árbol sin límite de profundidad y el árbol regularizado usado en la comparación principal (max_depth=12, min_samples_leaf=10). Un F1 bajo en entrenamiento y prueba con brecha pequeña es evidencia de underfitting; un F1 muy alto en entrenamiento y claramente menor en prueba indica overfitting. También se revisan las brechas de los otros clasificadores en la tabla general.

In [ ]:
# Árbol simple: referencia para identificar underfitting.
arbol_simple = Pipeline(steps=[
    ('preprocesamiento', preprocesador_clf),
    ('modelo', DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE))
])
arbol_simple.fit(X_train_clf, y_train_clf)

# Árbol sin poda: referencia para identificar overfitting.
arbol_sin_poda = Pipeline(steps=[
    ('preprocesamiento', preprocesador_clf),
    ('modelo', DecisionTreeClassifier(random_state=RANDOM_STATE))
])
arbol_sin_poda.fit(X_train_clf, y_train_clf)

predicciones_diagnostico = {
    'Decision Tree simple (max_depth=1)': (
        arbol_simple.predict(X_train_clf), arbol_simple.predict(X_test_clf)
    ),
    'Decision Tree sin poda': (
        arbol_sin_poda.predict(X_train_clf), arbol_sin_poda.predict(X_test_clf)
    ),
    'Decision Tree regularizado': (
        modelos_clasificacion['Decision Tree'].predict(X_train_clf),
        modelos_clasificacion['Decision Tree'].predict(X_test_clf)
    )
}

filas_diagnostico = []
for nombre, (pred_train, pred_test) in predicciones_diagnostico.items():
    f1_train = f1_score(y_train_clf, pred_train, average='macro', zero_division=0)
    f1_test = f1_score(y_test_clf, pred_test, average='macro', zero_division=0)
    filas_diagnostico.append({
        'modelo': nombre,
        'F1_macro_train': f1_train,
        'F1_macro_test': f1_test,
        'brecha_train_test': f1_train - f1_test
    })

tabla_diagnostico_arboles = pd.DataFrame(filas_diagnostico).set_index('modelo')
display(tabla_diagnostico_arboles.round(3))

# Agregar el árbol balanceado a la tabla general de métricas.
fila_balanceada = {'modelo': 'Decision Tree + RandomOverSampler'}
fila_balanceada.update({f'{k}_train': v for k, v in metricas_arbol_balanceado_train.items()})
fila_balanceada.update({f'{k}_test': v for k, v in metricas_arbol_balanceado_test.items()})
tabla_clasificacion_completa = pd.concat([
    tabla_clasificacion,
    pd.DataFrame([fila_balanceada]).set_index('modelo')
])
tabla_clasificacion_completa['brecha_f1_macro'] = (
    tabla_clasificacion_completa['f1_macro_train'] - tabla_clasificacion_completa['f1_macro_test']
)
tabla_clasificacion_completa = tabla_clasificacion_completa.join(tabla_cv_clasificacion, how='left')
display(tabla_clasificacion_completa.sort_values('f1_macro_test', ascending=False).round(3))

f1_simple_train = tabla_diagnostico_arboles.loc['Decision Tree simple (max_depth=1)', 'F1_macro_train']
f1_simple_test = tabla_diagnostico_arboles.loc['Decision Tree simple (max_depth=1)', 'F1_macro_test']
f1_sin_poda_train = tabla_diagnostico_arboles.loc['Decision Tree sin poda', 'F1_macro_train']
f1_sin_poda_test = tabla_diagnostico_arboles.loc['Decision Tree sin poda', 'F1_macro_test']
f1_regular_train = tabla_diagnostico_arboles.loc['Decision Tree regularizado', 'F1_macro_train']
f1_regular_test = tabla_diagnostico_arboles.loc['Decision Tree regularizado', 'F1_macro_test']

display(Markdown(
    f"**Underfitting:** el árbol simple obtuvo F1 macro de {f1_simple_train:.3f} en entrenamiento "
    f"y {f1_simple_test:.3f} en prueba. Ambos resultados son bajos y la brecha es pequeña, "
    f"lo que indica que su profundidad no alcanza para representar bien las diferencias entre clases.\n\n"
    f"**Overfitting:** el árbol sin poda obtuvo {f1_sin_poda_train:.3f} en entrenamiento y "
    f"{f1_sin_poda_test:.3f} en prueba, con una brecha de {f1_sin_poda_train-f1_sin_poda_test:.3f}. "
    f"El árbol regularizado redujo la brecha a {f1_regular_train-f1_regular_test:.3f} "
    f"({f1_regular_train:.3f} en entrenamiento y {f1_regular_test:.3f} en prueba)."
))

## 13. Importancia de variables

La importancia del árbol es un complemento descriptivo, no una prueba de causalidad ni la única regla para seleccionar atributos.

In [ ]:
arbol_entrenado = modelos_clasificacion['Decision Tree'].named_steps['modelo']
nombres_features = modelos_clasificacion['Decision Tree'].named_steps['preprocesamiento'].get_feature_names_out()
importancias = pd.Series(arbol_entrenado.feature_importances_, index=nombres_features).sort_values(ascending=False)
display(importancias.head(15).to_frame('importancia'))

plt.figure(figsize=(9, 5))
importancias.head(15).sort_values().plot(kind='barh')
plt.title('15 variables con mayor importancia en Decision Tree')
plt.xlabel('Importancia')
plt.ylabel('Feature transformado')
plt.tight_layout()
plt.show()

## 14. Selección del clasificador y análisis de métricas

La métrica prioritaria es **F1 macro**, porque las cuatro clases no tienen la misma frecuencia y el objetivo es evitar que el desempeño de Departamento oculte errores en Otros. Se revisan también accuracy, precision macro, recall macro, F1 ponderado, ROC-AUC One-vs-Rest y F1 macro de validación cruzada. Para seleccionar el modelo se considera competitivo aquel cuyo F1 macro de prueba quede a no más de 0,02 del máximo; entre esos candidatos se prioriza la menor brecha train-test y luego el mayor F1 macro medio de validación cruzada. Esta regla evita elegir automáticamente un modelo que memoriza el entrenamiento.

- **Accuracy:** proporción total de predicciones correctas. Es fácil de comunicar, pero puede ocultar el bajo rendimiento en clases minoritarias.
- **Precision:** reduce falsas alarmas al predecir una clase, pero puede bajar si el modelo intenta recuperar más casos.
- **Recall:** mide cuántos casos reales de una clase se detectan; es importante para no dejar propiedades minoritarias sin identificar, aunque puede aumentar falsos positivos.
- **F1:** combina precision y recall mediante media armónica. Resume el equilibrio, pero no muestra por sí sola cuál de las dos métricas es baja.
- **ROC-AUC:** evalúa la capacidad de ordenar correctamente las clases frente al resto a distintos umbrales. En multiclase se calcula One-vs-Rest; no reemplaza el análisis de la matriz de confusión.
- **Promedio macro:** todas las clases pesan igual, adecuado cuando interesa el desempeño en cada categoría. **Promedio weighted:** pondera por cantidad de casos y refleja más el resultado de la clase mayoritaria.

En el negocio, confundir una Casa con un Departamento puede afectar la categorización y búsqueda de una publicación. No detectar PH u Otros puede ocultar publicaciones que tienen características distintas a la clase mayoritaria. El balanceo mejora el recall de la clase minoritaria, pero debe evaluarse junto con precision y F1 para evitar aumentar demasiado los falsos positivos.

In [ ]:
# Comparación final: desempeño en prueba y validación cruzada.
columnas_finales = [
    'accuracy_test', 'precision_macro_test', 'recall_macro_test',
    'f1_macro_test', 'f1_weighted_test', 'roc_auc_ovr_macro_test',
    'F1_macro_validacion_CV_media', 'F1_macro_validacion_CV_std', 'brecha_f1_macro'
]
tabla_final_clf = tabla_clasificacion_completa[columnas_finales].sort_values(
    ['f1_macro_test', 'F1_macro_validacion_CV_media'], ascending=False
)
display(tabla_final_clf)

f1_maximo_test = tabla_final_clf['f1_macro_test'].max()
candidatos_competitivos = tabla_final_clf[
    tabla_final_clf['f1_macro_test'] >= f1_maximo_test - 0.02
]
mejor_clasificador = candidatos_competitivos.sort_values(
    ['brecha_f1_macro', 'F1_macro_validacion_CV_media'],
    ascending=[True, False]
).index[0]
fila_mejor_clf = tabla_final_clf.loc[mejor_clasificador]
print(
    f'F1 macro máximo en prueba = {f1_maximo_test:.3f}. '
    f'Candidatos dentro de 0,02 puntos: {list(candidatos_competitivos.index)}. '
    f'Se selecciona {mejor_clasificador} por su brecha train-test y contraste con CV. '
    f'F1 macro prueba = {fila_mejor_clf["f1_macro_test"]:.3f}; '
    f'Recall macro prueba = {fila_mejor_clf["recall_macro_test"]:.3f}; '
    f'ROC-AUC OVR macro = {fila_mejor_clf["roc_auc_ovr_macro_test"]:.3f}; '
    f'F1 macro medio en CV = {fila_mejor_clf["F1_macro_validacion_CV_media"]:.3f} '
    f'(desviación estándar = {fila_mejor_clf["F1_macro_validacion_CV_std"]:.3f}); '
    f'brecha train-test = {fila_mejor_clf["brecha_f1_macro"]:.3f}.'
)

## 15. Conclusiones y respuesta a la pregunta de negocio

Las conclusiones siguientes se generan con los resultados numéricos obtenidos en las celdas anteriores. Se comparan desempeño en prueba, brecha entre entrenamiento y prueba y validación cruzada. Las cifras describen esta muestra del dataset Properati y no representan una garantía de desempeño en publicaciones futuras.

In [ ]:
fila_cv_reg = tabla_cv_regresion.loc[nombre_mejor_regresion]
recall_otros_sin_balance = comparacion_balance.loc['Decision Tree sin balance', 'recall_Otros_test']
recall_otros_balance = comparacion_balance.loc['Decision Tree + RandomOverSampler', 'recall_Otros_test']
f1_balance = comparacion_balance.loc['Decision Tree + RandomOverSampler', 'f1_macro_test']

f1_knn_train = tabla_clasificacion.loc['KNN Classifier (k=7)', 'f1_macro_train']
f1_knn_test = tabla_clasificacion.loc['KNN Classifier (k=7)', 'f1_macro_test']
gap_knn = f1_knn_train - f1_knn_test

f1_sin_poda_train = tabla_diagnostico_arboles.loc['Decision Tree sin poda', 'F1_macro_train']
f1_sin_poda_test = tabla_diagnostico_arboles.loc['Decision Tree sin poda', 'F1_macro_test']

mae_lineal_test = tabla_regresion.loc['Regresión lineal múltiple', 'MAE_test_USD']
diferencia_mae = mae_lineal_test - mejor_regresion['MAE_test_USD']

display(Markdown(
    f"**Regresión.** Se selecciona **{nombre_mejor_regresion}**. En prueba obtuvo MAE de "
    f"**USD {mejor_regresion['MAE_test_USD']:,.2f}**, RMSE de **USD {mejor_regresion['RMSE_test_USD']:,.2f}** "
    f"y R² de **{mejor_regresion['R2_test']:.3f}**. Su MAE de entrenamiento fue USD "
    f"{mejor_regresion['MAE_train_USD']:,.2f}, con una brecha train-test de "
    f"USD {mejor_regresion['brecha_MAE_USD']:,.2f}. Frente a la regresión lineal, redujo el MAE "
    f"de prueba en **USD {diferencia_mae:,.2f}**. En validación cruzada, el MAE medio de validación fue "
    f"USD {fila_cv_reg['MAE_CV_validacion_media_USD']:,.2f} (desviación estándar "
    f"USD {fila_cv_reg['MAE_CV_validacion_std_USD']:,.2f}) y R² medio "
    f"{fila_cv_reg['R2_CV_validacion_media']:.3f}.\n\n"
    f"**Clasificación.** La regla definida selecciona **{mejor_clasificador}**: F1 macro de prueba "
    f"**{fila_mejor_clf['f1_macro_test']:.3f}**, recall macro **{fila_mejor_clf['recall_macro_test']:.3f}**, "
    f"ROC-AUC OVR macro **{fila_mejor_clf['roc_auc_ovr_macro_test']:.3f}** y F1 macro medio de "
    f"validación cruzada **{fila_mejor_clf['F1_macro_validacion_CV_media']:.3f}** "
    f"(desviación estándar {fila_mejor_clf['F1_macro_validacion_CV_std']:.3f}). "
    f"Su brecha F1 macro train-test fue {fila_mejor_clf['brecha_f1_macro']:.3f}.\n\n"
    f"**Balance de clases.** RandomOverSampler elevó el recall de Otros de {recall_otros_sin_balance:.3f} "
    f"a {recall_otros_balance:.3f}, pero el F1 macro del árbol bajó de "
    f"{tabla_clasificacion.loc['Decision Tree', 'f1_macro_test']:.3f} a {f1_balance:.3f}. "
    f"Por eso el balance resulta útil si el objetivo prioritario es detectar más casos Otros, "
    f"aunque aumenta las falsas alarmas y reduce el equilibrio general.\n\n"
    f"**Generalización.** KNN de clasificación mostró una brecha F1 macro de {gap_knn:.3f} "
    f"(entrenamiento {f1_knn_train:.3f}; prueba {f1_knn_test:.3f}), señal de overfitting. "
    f"El árbol sin poda también presentó una brecha de {f1_sin_poda_train-f1_sin_poda_test:.3f}. "
    f"El árbol simple de profundidad 1 evidencia underfitting porque su F1 fue bajo tanto en "
    f"entrenamiento como en prueba. La regularización reduce la brecha del árbol y entrega un "
    f"compromiso más equilibrado para este caso."
))

### Limitaciones

Las muestras controlan el tiempo de KNN/SVM. El precio publicado no necesariamente es el precio final de venta; la ubicación puede ser incompleta y los resultados corresponden a un periodo y mercado específicos. Se recomienda validar con publicaciones nuevas.

## 16. Checklist de la pauta de evaluación

| Indicador | Evidencia en el notebook |
|---|---|
| IE 1 - Target continuo | price, dtype, valores únicos y distribución |
| IE 2 - Modelos de regresión | LinearRegression y KNeighborsRegressor |
| IE 3 - Métricas de regresión | MAE, RMSE y R² en train/test, en USD |
| IE 4 - Selección de regresión | Tabla comparativa, MAE de prueba, brecha y validación cruzada |
| IE 5 - Diferencia regresión/clasificación | Explicación y revisión de ambos targets |
| IE 6 - Correlación | Pearson y Spearman, incluyendo rooms con bedrooms, heatmaps y relación con price |
| IE 7 - Transformación | One-Hot Encoding, imputación y escalado dentro de Pipeline |
| IE 8 - Selección de features | Justificación de variables conservadas y descartadas; exclusión de data leakage |
| IE 9 - Clasificación | Logistic Regression, KNN, SVM y Decision Tree |
| IE 10 - Generalización | Train/test, cross_validate, evidencia de overfitting y underfitting y regularización |
| IE 11 - Métricas de clasificación | Reporte por clase, macro, weighted y ROC-AUC OVR |
| IE 12 - Matriz de confusión | Matrices, errores principales por clase y cálculo de TP, FP y FN |
| IE 13 - Balance de clases | RandomOverSampler solo en train y comparación de recall/F1 |
| IE 14 - Selección de clasificador | Tabla comparativa y selección por F1 macro, brecha y CV |
| IE 15 - Ventajas y desventajas | Análisis de accuracy, precision, recall, F1, ROC-AUC, macro y weighted |

**Antes de entregar:** ejecutar todas las celdas desde el inicio, comprobar que no hay errores, revisar que las salidas coincidan con las conclusiones y que cada integrante pueda explicar las decisiones de modelado.